# **StatQuest Illustrated Guide to Neural Networks and AI — Summary**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Chapter 01: Fundamental Concepts in Neural Networks and AI](#Chapter-01:-Fundamental-Concepts-in-Neural-Networks-and-AI)
- [Chapter 02: Optimizing Weights and Biases with Backpropagation](#Chapter-02:-Optimizing-Weights-and-Biases-with-Backpropagation)
- [Chapter 03: Networks with Multiple Inputs and Outputs](#Chapter-03:-Networks-with-Multiple-Inputs-and-Outputs)
- [Chapter 04: Simplifying Outputs with ArgMax and SoftMax](#Chapter-04:-Simplifying-Outputs-with-ArgMax-and-SoftMax)
- [Chapter 05: Speeding up Training with Cross Entropy](#Chapter-05:-Speeding-up-Training-with-Cross-Entropy)
- [Chapter 06: Image Classification with Convolutional Neural Networks](#Chapter-06:-Image-Classification-with-Convolutional-Neural-Networks)
- [Chapter 07: Stock Prediction with Recurrent Neural Networks (RNNs)](#Chapter-07:-Stock-Prediction-with-Recurrent-Neural-Networks-(RNNs))
- [Chapter 08: Better Stock Prediction with Long Short-Term Memory (LSTM)](#Chapter-08:-Better-Stock-Prediction-with-Long-Short-Term-Memory-(LSTM))
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 01: Fundamental Concepts in Neural Networks and AI

Neural networks stretch, flip, crop and combine activation functions to create complex shapes that fits the data.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 02: Optimizing Weights and Biases with Backpropagation

**Backpropagation**
* Magnitude of derivative $\Longrightarrow$ How big of a step we should take towards the minimum
* Sign of derivative $\Longrightarrow$ direction of the step

**Evolution of Backpropagation Algorithms**
* **Gradient Descent:** All training data points used in each step of backpropagation for every weight & bias $\Longrightarrow$ too much computation required for each step
* **Stochastic Gradient Descent (SGD):** A random subset of data points used in each step of backpropagation for every weight & bias
* **Adam:** A little less stochastic than SGD by using a weighted average between each step

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 03: Networks with Multiple Inputs and Outputs

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 04: Simplifying Outputs with ArgMax and SoftMax

* ArgMax $\Longrightarrow$
    * Returns 1 for largest value (or values if there is a tie) and 0s for all the remaining values
    * Limitations:
        * ArgMax cannot be used to rank outputs
        * Cannot be used during training, can only be used during inference $\Rightarrow$ cannot work with Backpropagation ($\frac {d (\text{ArgMax})}{d (\text{Raw output})} = 0$)
* SoftMax $\Longrightarrow$
    * Normalizes all values to be between 0 and 1 such that they add upto 1 $\Rightarrow$ largest SoftMax value is always closest to 1
    * $\text{SoftMax}_{i}(values) = \frac {e^{value_i}}{\sum_{j=1}^{k} e^{value_j}}$
    * Solves both the limitations of ArgMax
        * SoftMax can be used to rank outputs
        * Can be used during training $\Rightarrow$ can work with Backpropagation

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 05: Speeding up Training with Cross Entropy

* A good loss function should be:
    * having large slope i.e. steep (should have large derivatives) when making bad predictions $\Rightarrow$ large steps in gradient descent
    * having low slope (small derivatives) when making good predictions $\Rightarrow$ small steps in gradient descent
* When our model uses SoftMax to convert raw outputs (logits) to final output probabilities:
    * Sum of Squared Residuals (SSR) is not a good loss function as its slope is more or less the same when predictions change from worst to good
    * Cross Entropy ($\text{CE} = -log (p_{\text{true class}})$)is best suited
    * Example: For a subset with 3 data points of Iris dataset (Setosa, Versicolor, Virginica):
        * $\text{CE_{Setosa}} = -log (p_{Setosa})$
        * $\text{CE_{Versicolor}} = -log (p_{Versicolor})$
        * $\text{CE_{Virginica}} = -log (p_{Virginica})$
        * $\text{Total CE Loss} = \text{CE_{Setosa}} + \text{CE_{Versicolor}} + \text{CE_{Virginica}}$
        * During backpropagation, we reduce this loss


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 06: Image Classification with Convolutional Neural Networks

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 07: Stock Prediction with Recurrent Neural Networks (RNNs)

* Fully connected neural networks & CNNs $\Longrightarrow$ no of inputs is fixed
* Variable no of inputs $\Longrightarrow$ Recurrent Neural Networks (RNNs):
    * Each input (or time step) is processed by a neural net with feedback
    * RNN unrolls for each extra input (or time step) where output of activation is fed back to the next stage (with a weight $W_2$)
* Example for RNN: Stock Price Prediction
    * Predict tomorrow's price based on only today's price $\Longrightarrow$ No need for RNN to unroll; RNN works as a simple neural net with no feedback
    * Predict tomorrow's price based on today & yesterday $\Longrightarrow$ RNN unrolls once; yesterday term multiplied by $W_2$ in the gradient
    * Predict tomorrow's price based on today, yesterday & day before yesterday $\Longrightarrow$ RNN unrolls twice; yesterday term multiplied by $W_2^2$ in the gradient
    * Predict tomorrow's price based on today & 20 past time steps $\Longrightarrow$ RNN unrolls 20 times; yesterday term multiplied by $W_2^{20}$ in the gradient $\Longrightarrow$ can lead to Vanishing ($-1 < W_2 < 1$) or Exploding ($W_2 < -1$ or $W_2 > 1$) Gradients

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chapter 08: Better Stock Prediction with Long Short-Term Memory (LSTM)

* LSTM has multiple units for multiple time steps. Each unit has:
    * Inputs:
        * Long-term memory from previous unit (cell state)
        * Short-term memory from previous unit (hidden state) $\Rightarrow$ predictions
    * Forget gate:
        * Prev short-term memory * $W_{F1}$ + Current input * $W_{F2}$ + $B_{F1}$ $\Rightarrow$ Sigmoid $\Rightarrow$ Value between 0 & 1
        * Value between 0 & 1 x Prev Long-term memory = % of Long-term memory to keep (Scaled Long-term memory)
    * Input gate:
        * Prev short-term memory * $W_{I1}$ + Current input * $W_{I2}$ + $B_{I1}$ $\Rightarrow$ Tanh $\Rightarrow$ Value between -1 & 1 (Potential Long-term memory)
        * Prev short-term memory * $W_{I3}$ + Current input * $W_{I4}$ + $B_{I2}$ $\Rightarrow$ Sigmoid $\Rightarrow$ Value between 0 & 1
        * Value between 0 & 1 x Potential Long-term memory = % of Potential Long-term memory
    * New Long-term memory = Scaled Long-term memory + Potential Long-term memory
    * Output gate:
        * New Long-term memory $\Rightarrow$ Tanh $\Rightarrow$ Value between -1 & 1 (Potential Short-term memory)
        * Prev short-term memory * $W_{O1}$ + Current input * $W_{O2}$ + $B_{O1}$ $\Rightarrow$ Sigmoid $\Rightarrow$ Value between 0 & 1 (% of Potential Short-term memory to keep)
        * Value between 0 & 1 x Potential Short-term memory = % of Potential Short-term memory to keep i.e. New Short-term memory
* By using separate paths for Long-term memories & Short-term memories, LSTMs reduced the effects of Vanishing/Exploding Gradients problem $\Longrightarrow$ we can unroll them more times for sequences with longer inputs compared to basic RNNs

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[StatQuest Illustrated Guide to Neural Networks and AI](https://statquest.gumroad.com/l/kihdi)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)